# 02_core_functions
Three clearly separated sections:

1. **COMMON** - used by both pipelines: control rows, logging, and the *copy -> unzip* stages
2. **NPHC** - monthly: ingest one metadata row per JSON file (functions start with `nphc_`)
3. **HSAR** - daily: Auto Loader per dated (mid date) folder (functions start with `hsar_`)

# 1. COMMON

In [0]:
def get_active_control_entries(src_system):
    return spark.sql(f"""
        SELECT s_no, src_system, zip_source, zip_source_path, zip_tgt_path, unzip_tgt_path,
               tgt_catalog, tgt_schema, tgt_table, is_full_load, is_incremental_load
        FROM {control_table}
        WHERE is_active = 1 AND src_system = '{src_system}'
        ORDER BY s_no
    """).collect()


def write_logs(log_records, run_id):
    if not log_records:
        return
    rows = [
        (run_id, r.get("stage"),
         int(r["s_no"]) if r.get("s_no") is not None else None,
         r.get("source"), r.get("file_name"), r.get("source_path"), r.get("target_path"),
         r.get("status"), r.get("message"), r.get("start_time"), r.get("end_time"))
        for r in log_records
    ]
    spark.createDataFrame(rows, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(log_table)

## Copy -> Unzip (both pipelines)
Both sources use the same zip name layout. The **MID date** is the folder name (`raw/<mid>/`, `processed/<mid>/`); the **END date** is what the daily/monthly check looks at.

Which zips are copied:

- `is_incremental_load = 1` (default): only **new** zips, meaning zips not already in `<zip_tgt_path>/<date>/`. The first run copies everything; a missed month/day is still "new" next time and is picked up automatically.
- `is_full_load = 1`: copy **every** zip again.

In [0]:
def parse_file_name(file_name):
    # 'case_data_001_20220605_20220620_20260101_235954.zip'
    #  -> {'table_name': 'case_data_001', 'start_date': '20220605', 'mid_date': '20220620', 'end_date': '20260101'}
    # Returns None if the name does not follow the expected pattern.
    m = FILE_NAME_PATTERN.match(file_name)
    return m.groupdict() if m else None


def zip_folder_date(file_name):
    # MID date: the raw/<date>/ and processed/<date>/ folder (both pipelines); None if the name does not match
    parts = parse_file_name(file_name)
    return parts["mid_date"] if parts else None


def nphc_months(zip_files):
    # yyyymm taken from the END date, e.g. ['202604', '202605']
    return sorted({d[:6] for d in zip_end_dates(zip_files)})


def zip_end_dates(zip_files):
    # END dates (yyyymmdd) of the given zips, sorted: used to check what is new (NPHC by month, HSAR by day)
    return sorted({p["end_date"] for p in (parse_file_name(f.name) for f in zip_files) if p})


def list_zip_files(source_path):
    source_path = source_path.rstrip("/") + "/"
    return [f for f in dbutils.fs.ls(source_path) if not f.isDir() and f.name.lower().endswith(".zip")]


def raw_zip_path(file_name, target_base_path):
    # <zip_tgt_path>/<mid_date>/<file_name>, or None if the name does not match the pattern
    folder_date = zip_folder_date(file_name)
    if folder_date is None:
        return None
    return f"{target_base_path.rstrip('/')}/{folder_date}/{file_name}"


def select_zips_to_copy(zip_files, ctl):
    if ctl["is_full_load"] == 1:
        return zip_files
    if ctl["is_incremental_load"] == 1:
        new = []
        for f in zip_files:
            raw_path = raw_zip_path(f.name, ctl["zip_tgt_path"])
            if raw_path is None or not os.path.exists(raw_path):      # bad names are kept so they get logged as SKIPPED
                new.append(f)
        return new
    return []


def discard_raw_copy(path):
    # after a failed unzip/load, remove the raw copy so the next run treats the zip as new and retries it
    try:
        dbutils.fs.rm(path)
    except Exception as e:
        print(f"WARNING: could not remove {path}: {e}")


def copy_zip(file_info, ctl):
    # copy one zip into <zip_tgt_path>/<date>/ and return a log record
    start_time = datetime.now()
    record = {"s_no": ctl["s_no"], "stage": "COPY", "source": ctl["src_system"], "file_name": file_info.name,
              "source_path": file_info.path, "target_path": None, "start_time": start_time}

    target_file = raw_zip_path(file_info.name, ctl["zip_tgt_path"])
    if target_file is None:
        record.update(status="SKIPPED", message="File name does not match <name>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.zip",
                      end_time=datetime.now())
        return record

    record["target_path"] = target_file
    try:
        dbutils.fs.mkdirs(target_file.rsplit("/", 1)[0] + "/")
        dbutils.fs.cp(file_info.path, target_file)
        record.update(status="SUCCESS", message="Copied successfully")
    except Exception as e:
        record.update(status="FAILED", message=str(e))
    record["end_time"] = datetime.now()
    return record


def unzip_zip(copy_record, unzip_base_path):
    # unzip one copied file into <unzip_tgt_path>/<date>/<zip_name_without_.zip>/
    # (own sub-folder per zip, so files from different zips never overwrite each other)
    start_time = datetime.now()
    file_name = copy_record["file_name"]
    zip_path  = copy_record["target_path"]
    record = {"s_no": copy_record["s_no"], "stage": "UNZIP", "source": copy_record["source"],
              "file_name": file_name, "source_path": zip_path, "target_path": None, "start_time": start_time}

    folder_date   = zip_folder_date(file_name)
    target_folder = f"{unzip_base_path.rstrip('/')}/{folder_date}/{file_name.rsplit('.', 1)[0]}/"
    record["target_path"] = target_folder
    try:
        dbutils.fs.mkdirs(target_folder)
        with zipfile.ZipFile(zip_path.replace("dbfs:", ""), "r") as z:
            z.extractall(target_folder)
        record.update(status="SUCCESS", message="Extracted successfully")
    except zipfile.BadZipFile:
        record.update(status="FAILED", message="Invalid ZIP file")
    except Exception as e:
        record.update(status="FAILED", message=str(e))
    record["end_time"] = datetime.now()
    return record


def copy_and_unzip(run_id, control_entries, log_records):
    # runs Stage 1 (copy) and Stage 2 (unzip) for the given control rows, appends to log_records and writes the logs
    ctl_by_sno = {c["s_no"]: c for c in control_entries}

    print("=== Stage 1 - Copy ===")
    for ctl in control_entries:
        all_zips = list_zip_files(ctl["zip_source_path"])
        zips = select_zips_to_copy(all_zips, ctl)
        print(f"s_no={ctl['s_no']} | {ctl['zip_source_path']}: {len(all_zips)} zip file(s) at source, {len(zips)} to copy")
        for f in zips:
            rec = copy_zip(f, ctl)
            log_records.append(rec)
            print(f"  [COPY:{rec['status']}] {rec['file_name']}: {rec['message']}")

    print("=== Stage 2 - Unzip ===")
    copied = [r for r in log_records if r["stage"] == "COPY" and r["status"] == "SUCCESS"]
    print(f"{len(copied)} zip file(s) to unzip")
    for copy_rec in copied:
        rec = unzip_zip(copy_rec, ctl_by_sno[copy_rec["s_no"]]["unzip_tgt_path"])
        log_records.append(rec)
        print(f"  [UNZIP:{rec['status']}] {rec['file_name']}: {rec['message']}")
        if rec["status"] == "FAILED":
            discard_raw_copy(copy_rec["target_path"])        # retried on the next run

    write_logs(log_records, run_id)
    print(f"Logged {len(log_records)} COPY/UNZIP record(s) to {log_table}")

# 2. NPHC (monthly) - ingest JSON
One metadata row per JSON file (schema captured as `VARIANT`) plus the **`zip_file`** it came from.

In [0]:
def nphc_create_target_table(target):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target} (
        src_system      STRING,
        file_id         BIGINT,      -- serial no.
        zip_file        STRING,      -- zip the JSON file came from
        file_name       STRING,
        file_date       TIMESTAMP,   -- modified time of the extracted file
        file_path       STRING,
        dt_datafile     STRING,      -- middle date from file name (yyyymmdd)
        file_size_mb    DOUBLE,
        file_structure  VARIANT,
        table_name      STRING,      -- from file name
        created_by      STRING,
        created_on      TIMESTAMP
    ) USING DELTA
    """)
    # table created by an earlier version of the pipeline? add the new column
    if "zip_file" not in spark.table(target).columns:
        spark.sql(f"ALTER TABLE {target} ADD COLUMNS (zip_file STRING AFTER file_id)")


def nphc_list_json_files(path):
    # all .json files under path, including sub-folders
    out = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            out.extend(nphc_list_json_files(item.path))
        elif item.name.lower().endswith(".json"):
            out.append(item)
    return sorted(out, key=lambda x: x.path)


def nphc_log_load_event(run_id, ctl, zip_file, f, target, status, started, error=None):
    # append one row to the load log; logging must never break the load
    try:
        row = (
            run_id, ctl["src_system"], ctl["unzip_tgt_path"], zip_file,
            f.name if f else None, f.path if f else None,
            f.size / (1024.0 * 1024.0) if f else None,
            int(f.modificationTime) if f else None,
            target, status,
            type(error).__name__ if error else None,
            (str(error)[:4000] + "\n" + traceback.format_exc()[-2000:]) if error else None,
            started, datetime.now(), EXECUTED_BY,
        )
        spark.createDataFrame([row], LOAD_LOG_SCHEMA).write.mode("append").saveAsTable(load_log_table)
    except Exception as log_err:
        print(f"WARNING: could not write load log row: {log_err}")


def nphc_load_file(f, target, zip_file, src_system):
    # read one JSON file, capture its structure, write one metadata row to target
    parts = parse_file_name(f.name)
    if parts is None:
        raise ValueError(f"Bad file name '{f.name}'. Expected <table>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.json")
    dt_datafile = datetime.strptime(parts["mid_date"], "%Y%m%d").strftime("%Y%m%d")

    file_structure = spark.read.option("multiLine", "true").json(f.path).schema.json()

    # replace any row from an earlier run of the same file, so re-runs never duplicate
    spark.sql(f"DELETE FROM {target} WHERE file_path = :p", args={"p": f.path})
    file_id = spark.sql(f"SELECT COALESCE(MAX(file_id), 0) + 1 FROM {target}").first()[0]

    (spark.createDataFrame([Row(
            src_system     = src_system,
            file_id        = file_id,
            zip_file       = zip_file,
            file_name      = f.name,
            file_date      = datetime.fromtimestamp(f.modificationTime / 1000),
            file_path      = f.path,
            dt_datafile    = dt_datafile,
            file_size_mb   = f.size / (1024.0 * 1024.0),
            file_structure = file_structure,
            table_name     = parts["table_name"],
            created_by     = EXECUTED_BY,
        )])
        .withColumn("file_structure", F.parse_json("file_structure"))
        .withColumn("created_on", F.current_timestamp())
        .select("src_system", "file_id", "zip_file", "file_name", "file_date", "file_path", "dt_datafile",
                "file_size_mb", "file_structure", "table_name", "created_by", "created_on")
        .write.mode("append").saveAsTable(target))


def nphc_ingest_zip(unzip_record, ctl, run_id):
    # ingest every JSON file extracted from one zip. Returns {'SUCCESS': n, 'FAILED': n}
    target   = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    zip_file = unzip_record["file_name"]
    counts   = {"SUCCESS": 0, "FAILED": 0}

    started = datetime.now()
    try:
        nphc_create_target_table(target)
        files = nphc_list_json_files(unzip_record["target_path"])
        if not files:
            raise ValueError(f"No .json files found in {unzip_record['target_path']}")
    except Exception as e:
        nphc_log_load_event(run_id, ctl, zip_file, None, target, "FAILED", started, e)
        counts["FAILED"] += 1
        print(f"  [INGEST:FAILED] {zip_file}: {str(e)[:300]}")
        return counts

    for f in files:
        started = datetime.now()
        try:
            nphc_load_file(f, target, zip_file, ctl["src_system"])
            nphc_log_load_event(run_id, ctl, zip_file, f, target, "SUCCESS", started)
            counts["SUCCESS"] += 1
            print(f"  [INGEST:SUCCESS] {f.name}")
        except Exception as e:
            nphc_log_load_event(run_id, ctl, zip_file, f, target, "FAILED", started, e)
            counts["FAILED"] += 1
            print(f"  [INGEST:FAILED] {f.name}: {str(e)[:300]}")
    return counts

# 3. HSAR (daily) - Auto Loader

After copy -> unzip, each dated folder `<unzip_tgt_path>/<mid_date>/` is loaded with Auto Loader: every JSON file becomes one row (raw text + parsed `VARIANT`), with its own checkpoint at `<unzip_tgt_path>/checkpoint/<yyyymmdd>`.

Only the dated folders that received new zips in this run are loaded. If a second zip lands in a day that is already loaded, the checkpoint makes Auto Loader pick up only the new files.

`is_full_load = 1` clears a folder's checkpoint and rows first, so a full reload never duplicates.

In [0]:
def hsar_checkpoint(ctl, dated_folder):
    return f"{ctl['unzip_tgt_path'].rstrip('/')}/checkpoint/{dated_folder}"


def hsar_reset_folder(ctl, dated_folder):
    # forget a dated folder: remove its checkpoint and any rows already loaded from it
    target = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    dbutils.fs.rm(hsar_checkpoint(ctl, dated_folder), True)
    if spark.catalog.tableExists(target):
        spark.sql(f"DELETE FROM {target} WHERE source_folder = '{dated_folder}'")


def hsar_load_folder(ctl, dated_folder):
    # load one dated folder from processed/ into the target table. Returns how many files that folder has in the target
    target      = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    folder_path = f"{ctl['unzip_tgt_path'].rstrip('/')}/{dated_folder}/"

    m       = F.col("_metadata")
    name_dt = F.regexp_extract(m["file_name"], HSAR_FILE_DATE_PATTERN, 1)
    dt_file = F.when(name_dt != "", name_dt).otherwise(None)
    # <processed>/<yyyymmdd>/<zip_name>/<file>.json  ->  <zip_name>.zip
    zip_file = F.concat(F.regexp_extract(m["file_path"], r"/\d{8}/([^/]+)/", 1), F.lit(".zip"))

    raw = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("wholetext", "true")
        .option("pathGlobFilter", "*.json")
        .option("cloudFiles.maxFilesPerTrigger", HSAR_MAX_FILES_PER_TRIGGER)
        .load(folder_path)
    )

    df = (
        raw.select(
            F.lit(ctl["src_system"]).alias("src_system"),
            zip_file.alias("zip_file"),
            F.lit(folder_path).alias("file_path"),
            F.lit(dated_folder).alias("source_folder"),
            m["file_name"].alias("file_name"),
            F.col("value").alias("json_original_data"),
            F.expr("try_parse_json(value)").alias("json_data_for_parsing"),
            (m["file_size"] / 1024.0).alias("file_size_kb"),
            m["file_modification_time"].alias("file_modified_date"),
            dt_file.alias("dt_data_file"),
        )
        .withColumn("dt_data_file_date", F.to_date("dt_data_file", "yyyyMMdd"))
        .withColumn("data_year", F.year("dt_data_file_date"))
        .withColumn("data_month", F.month("dt_data_file_date"))
        .withColumn("loaded_at", F.current_timestamp())
    )

    q = (
        df.writeStream.format("delta")
        .option("checkpointLocation", hsar_checkpoint(ctl, dated_folder))
        .trigger(availableNow=True)
        .partitionBy("data_year", "data_month")
        .toTable(target)
    )
    q.awaitTermination()
    # count from the target table (streaming progress metrics can be None and must never fail a load that worked)
    return spark.table(target).filter(F.col("source_folder") == dated_folder).count()